<a href="https://colab.research.google.com/github/Andresflorezdev/DAT-05_ANALISIS_y_Visualizacion_de_Datos-Esumer-Medellin/blob/main/Laboratorio_Clase_8_Estudiantes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase 8 · Transformación y enriquecimiento de datos
**Analítica y Visualización de Datos para la Toma de Decisiones · ESUMER / Estud-IA**  
Docente: Johan Steev Muriel Mosquera

**Objetivo de aprendizaje:** comprender cómo las transformaciones de columnas, las reglas de categorización y el tratamiento temporal cambian la representación y el significado analítico de los datos, y aplicar esos principios en un proceso reproducible.

## 0. Conexión a Google Drive
Esta es la primera celda de código. Ejecútala y autoriza la conexión en Colab. Antes de la carga crea en **Mi unidad** la carpeta `Curso_Analitica/Clase_8` y coloca allí `Matriculas_diagnostico.csv`.
Guardar el notebook en Drive y montar Drive son acciones diferentes: el montaje permite leer y escribir archivos desde Python.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
ruta_csv = '/content/drive/MyDrive/Analisis de Datos Esumer/Clase 8/Matriculas_diagnostico.csv' #Agregar la ruta del archivo en drive

## Cómo trabajar
Ejecuta las celdas en orden. La preparación está resuelta; los ejercicios de transformación requieren tus decisiones y código. Cada ejercicio incluye una meta, una pista conceptual, una ayuda de sintaxis desplegable y una comprobación independiente.

Si una comprobación falla, lee su mensaje, revisa la celda anterior y vuelve a ejecutar desde allí. Al reiniciar el entorno, monta Drive y ejecuta todo nuevamente. Conserva una copia del notebook con tu nombre. Puedes completar el recorrido a tu ritmo: todos los ejercicios forman parte del mismo laboratorio.

**Caso:** una institución ficticia quiere describir sus matrículas por edades, descuentos y anticipación. Los datos son sintéticos. Cada fila representa **una matrícula**, no una persona única. Los resultados describen matrículas.

**Ruta de 40 minutos:** preparación (5), cálculos (8), categorías (7), fechas (10), validación e interpretación (10). Los tiempos orientan el avance, no eliminan actividades.

In [7]:
from pathlib import Path
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj.to_string() if hasattr(obj, 'to_string') else obj)
ARCHIVO = Path(ruta_csv)
if not ARCHIVO.exists():
    raise FileNotFoundError(f'No encontramos {ARCHIVO}. Revisa el montaje, la carpeta, el nombre y la extensión del CSV.')
df_original = pd.read_csv(ARCHIVO, encoding='utf-8-sig',
                          dtype={'id_matricula':'string','id_estudiante':'string'})
print('Pandas:', pd.__version__, '| Fuente:', df_original.shape)
display(df_original.head(3))

Pandas: 2.2.3 | Fuente: (123, 12)


,id_matricula,id_estudiante,periodo,programa,sede,edad,fecha_matricula,fecha_inicio,estado,valor_matricula,descuento_pct,beca
0,M0001,E001,2026-1,analítica,Robledo,58,2026-01-08,2026-01-25,Activa,1200000,20.0,Sí
1,M0002,E002,2026-1,MARKETING,Centro,33,2026-01-12,2026-01-25,Cancelada,1400000,0.0,No
2,M0003,E003,2026-1,Desarrollo Web,Virtual,NaN,2026-01-07,2026-01-25,Finalizada,1800000,0.0,No


## 1. Preparación esperada del taller anterior
Este bloque reproduce la limpieza anterior. **No lo reemplaces por `dropna()` o una eliminación de duplicados de estudiante.**

| Campo o problema | Regla de referencia |
|---|---|
| `id_matricula` / `id_estudiante` | Identificadores de texto. Un estudiante puede tener varias matrículas. |
| Copias exactas / llave contradictoria | Conservar una copia exacta y separar todas las versiones de llaves contradictorias. |
| Programa | Homologación confirmada: Mercadeo = Marketing; Administracion = Administración. |
| Edad | Válida de 16 a 80 años. Valores inválidos quedan desconocidos. |
| Valor de matrícula | COP enteros, positivos. En esta fuente los puntos separan miles. |
| Descuento | De 0 a 100 puntos porcentuales. Un faltante no equivale a cero. |
| Fechas | Formato ISO. Matrícula anterior o igual al inicio. Conservar y marcar contradicciones. |
| Beca | Sí / No. Un faltante es desconocido. No inferir un descuento adicional. |

Las reglas de rango son ficticias y propias del caso. Conservamos valores fuente y banderas de revisión.

In [8]:
def preparar_matriculas(df):
    """Reglas de preparación del taller anterior; conserva la fuente."""
    # Eliminación controlada de copias exactas. La confirmación del caso permite hacerlo.
    copias_exactas = df.loc[df.duplicated()].copy()
    trabajo = df.drop_duplicates().copy()
    conflicto_llave = trabajo.duplicated('id_matricula', keep=False)
    pendientes_llave = trabajo.loc[conflicto_llave].copy()
    limpio = trabajo.loc[~conflicto_llave].copy()
    limpio['programa_fuente'] = limpio['programa']
    mapa_programas = {'analítica':'Analítica','marketing':'Marketing','mercadeo':'Marketing',
     'desarrollo web':'Desarrollo Web','administración':'Administración',
     'administracion':'Administración','especialización':'Especialización'}
    limpio['programa'] = limpio['programa'].astype('string').str.strip().str.lower().map(mapa_programas)
    limpio['revisar_programa'] = limpio['programa'].isna()
    for col in ['sede','estado']:
        limpio[col+'_fuente'] = limpio[col]
        limpio[col] = limpio[col].astype('string').str.strip().str.title()
    limpio['edad_fuente'] = limpio['edad']
    limpio['edad'] = pd.to_numeric(limpio['edad'], errors='coerce')
    limpio['revisar_edad'] = limpio['edad'].isna() | ~limpio['edad'].between(16,80)
    limpio.loc[limpio['revisar_edad'], 'edad'] = float('nan')
    limpio['valor_matricula_fuente'] = limpio['valor_matricula']
    valor_texto = limpio['valor_matricula'].astype('string')
    valor_texto = valor_texto.str.replace('$','',regex=False).str.replace(' ','',regex=False).str.replace('.','',regex=False)
    valor_convertido = pd.to_numeric(valor_texto, errors='coerce')
    limpio['revisar_valor'] = valor_convertido.isna() | valor_convertido.le(0)
    limpio['valor_matricula'] = valor_convertido.mask(limpio['revisar_valor'])
    limpio['descuento_pct_fuente'] = limpio['descuento_pct']
    limpio['descuento_pct'] = pd.to_numeric(limpio['descuento_pct'], errors='coerce')
    limpio['revisar_descuento'] = limpio['descuento_pct'].isna() | ~limpio['descuento_pct'].between(0,100)
    limpio.loc[limpio['revisar_descuento'], 'descuento_pct'] = float('nan')
    for col in ['fecha_matricula','fecha_inicio']:
        limpio[col+'_fuente'] = limpio[col]
        limpio[col] = pd.to_datetime(limpio[col], format='%Y-%m-%d', errors='coerce')
    limpio['revisar_fecha'] = (limpio['fecha_matricula'].isna() | limpio['fecha_inicio'].isna()
                             | limpio['fecha_matricula'].gt(limpio['fecha_inicio']))
    # Se conserva la fecha contradictoria y se excluye de análisis de plazos hasta validarla.
    limpio['revisar_beca'] = limpio['beca'].isna()
    return limpio, copias_exactas, pendientes_llave

In [9]:
base, copias, pendientes = preparar_matriculas(df_original)
assert len(df_original) == len(base) + len(copias) + len(pendientes)
assert len(base) == 119 and len(copias) == 2 and len(pendientes) == 2
assert base['id_matricula'].is_unique
banderas = ['revisar_programa','revisar_edad','revisar_valor','revisar_descuento','revisar_fecha','revisar_beca']
display(base[banderas].sum().rename('Incidencias pendientes'))
print('Filas afectadas:', base[banderas].any(axis=1).sum())
analitica = base.copy(deep=True)
base_control = base.copy(deep=True)

,Incidencias pendientes
revisar_programa,1
revisar_edad,12
revisar_valor,3
revisar_descuento,3
revisar_fecha,3
revisar_beca,1


Filas afectadas: 23


## 2. Nombres y columnas calculadas
### Ejercicio 1 · Un nombre que explique el dato
Renombra únicamente `valor_matricula` como `valor_bruto_matricula`. Conserva `valor_matricula_fuente` para la trazabilidad.

**Pista conceptual:** renombrar cambia una etiqueta, no el contenido de la columna. `rename()` devuelve un DataFrame: conserva ese resultado.
<details><summary>Ayuda de sintaxis</summary>

`tabla = tabla.rename(columns={'nombre_actual': 'nombre_nuevo'})`
</details>

In [14]:
# COMPLETA AQUÍ
# Renombra y conserva el resultado en analitica.
analitica = analitica.rename(columns={
    'valor_matricula': 'valor_bruto_matricula'
})

In [15]:
assert 'valor_bruto_matricula' in analitica.columns, 'Conserva el resultado de rename().'
assert 'valor_matricula' not in analitica.columns
assert 'valor_matricula_fuente' in analitica.columns
pd.testing.assert_series_equal(analitica['valor_bruto_matricula'], base['valor_matricula'], check_names=False)
print('Nombre actualizado; valores conservados.')

Nombre actualizado; valores conservados.


### Ejercicio 2 · Cálculos vectorizados con insumos pendientes
Crea `valor_descuento` y `valor_neto_matricula`. El descuento está expresado en puntos porcentuales: 20 significa 20 %. Redondea los valores calculados a dos decimales.

**Decisión:** cuando falte el valor bruto o el porcentaje, el resultado debe quedar desconocido. El neto es un cálculo después del descuento registrado, **no recaudo efectivo**. No uses `beca` para sumar otro descuento.

**Pista conceptual:** puedes operar directamente entre Series. Los nulos deben propagarse.
<details><summary>Ayuda de sintaxis</summary>

`tabla['nueva'] = (tabla['columna_a'] * tabla['columna_b'] / 100).round(2)`
</details>

In [16]:
# COMPLETA AQUÍ
# Crea las dos columnas sin rellenar insumos desconocidos.
analitica['valor_descuento'] = (analitica['valor_bruto_matricula'] * analitica['descuento_pct'] / 100).round(2)
analitica['valor_neto_matricula'] = (analitica['valor_bruto_matricula'] - analitica['valor_descuento']).round(2)

In [17]:
for c in ['valor_descuento','valor_neto_matricula']:
    assert c in analitica, f'Falta crear {c}.'
sin_insumos = analitica['valor_bruto_matricula'].isna() | analitica['descuento_pct'].isna()
assert analitica.loc[sin_insumos,'valor_neto_matricula'].isna().all(), 'Un resultado calculado no puede inventar un insumo.'
esperado = (base['valor_matricula'] * base['descuento_pct'] / 100).round(2)
pd.testing.assert_series_equal(analitica['valor_descuento'], esperado, check_names=False)
pd.testing.assert_series_equal(analitica['valor_neto_matricula'], (base['valor_matricula']-esperado).round(2), check_names=False)
display(analitica.loc[analitica['id_matricula'].isin(['M0001','M0002','M0008','M0023']),
 ['id_matricula','valor_bruto_matricula','descuento_pct','valor_descuento','valor_neto_matricula']])
print('Comprobación monetaria correcta. Neto pendiente:', analitica['valor_neto_matricula'].isna().sum())

,id_matricula,valor_bruto_matricula,descuento_pct,valor_descuento,valor_neto_matricula
0,M0001,1200000,20.0,240000.0,960000.0
1,M0002,1400000,0.0,0.0,1400000.0
7,M0008,<NA>,0.0,<NA>,<NA>
22,M0023,1600000,30.0,480000.0,1120000.0


Comprobación monetaria correcta. Neto pendiente: 6


## 3. Categorías con límites explícitos
### Ejercicio 3 · Rangos de edad
Construye `rango_edad` con etiquetas `16–24`, `25–34`, `35–44`, `45–59`, `60–80` y `Sin información`.

Usa intervalos cerrados a la izquierda (`right=False`) con cortes **16, 25, 35, 45, 60, 81**. Así, 25 pertenece a 25–34 y 80 sigue dentro del último rango. Este caso contiene edades enteras válidas.

**Pista conceptual:** los cortes son fronteras, no etiquetas. `pd.cut()` devuelve categorías; puedes convertir el resultado a `string` antes de asignar «Sin información».
<details><summary>Ayuda de sintaxis</summary>

`pd.cut(serie, bins=cortes, labels=etiquetas, right=False).astype('string').fillna('Sin información')`
</details>

In [18]:
# COMPLETA AQUÍ
CORTES_EDAD = [16,25,35,45,60,81]
ETIQUETAS_EDAD = ['16–24','25–34','35–44','45–59','60–80']
# Crea rango_edad.
analitica['rango_edad'] = pd.cut(analitica['edad'], bins=CORTES_EDAD, labels=ETIQUETAS_EDAD, right=False).astype('string').fillna('Sin información')

In [19]:
assert 'rango_edad' in analitica
esperado_edad = pd.cut(base['edad'], [16,25,35,45,60,81],
    labels=['16–24','25–34','35–44','45–59','60–80'], right=False).astype('string').fillna('Sin información')
pd.testing.assert_series_equal(analitica['rango_edad'], esperado_edad, check_names=False)
assert analitica['rango_edad'].eq('Sin información').sum() == 12
prueba_limites = pd.Series([16,24,25,34,35,44,45,59,60,80,float('nan')])
resultado_limites = pd.cut(prueba_limites,CORTES_EDAD,labels=ETIQUETAS_EDAD,right=False).astype('string').fillna('Sin información')
display(pd.DataFrame({'edad':prueba_limites,'categoria':resultado_limites}))
assert resultado_limites.tolist() == ['16–24','16–24','25–34','25–34','35–44','35–44','45–59','45–59','60–80','60–80','Sin información']
display(analitica['rango_edad'].value_counts(dropna=False))

,edad,categoria
0,16.0,16–24
1,24.0,16–24
2,25.0,25–34
3,34.0,25–34
4,35.0,35–44
5,44.0,35–44
6,45.0,45–59
7,59.0,45–59
8,60.0,60–80
9,80.0,60–80


,count
rango_edad,
45–59,36
25–34,25
35–44,24
16–24,22
Sin información,12


## 4. Fechas, duración y periodos
### Ejercicio 4 · Anticipación con validez
Crea `dias_anticipacion` restando `fecha_matricula` de `fecha_inicio` y extrayendo días con `.dt.days`. Usa `mask()` para dejar desconocidos los resultados de filas marcadas con `revisar_fecha`.

**Reto:** M0034 tiene dos fechas que Python puede interpretar, pero la matrícula ocurre después del inicio. Convertir correctamente no resuelve esa contradicción.
<details><summary>Ayuda de sintaxis</summary>

`duracion = (tabla['fin'] - tabla['inicio']).dt.days`  
`tabla['duracion'] = duracion.mask(tabla['bandera_revision'])`
</details>

In [20]:
# COMPLETA AQUÍ
# Calcula días y excluye resultados de fechas pendientes de validación.
duracion = (analitica['fecha_inicio'] - analitica['fecha_matricula']).dt.days
analitica['dias_anticipacion'] = duracion.mask(analitica['revisar_fecha'])

In [21]:
assert 'dias_anticipacion' in analitica
esperado_dias = (base['fecha_inicio']-base['fecha_matricula']).dt.days.mask(base['revisar_fecha'])
pd.testing.assert_series_equal(analitica['dias_anticipacion'],esperado_dias,check_names=False)
assert analitica['dias_anticipacion'].isna().sum() == 3
display(analitica.loc[analitica['id_matricula'].isin(['M0001','M0031','M0034','M0037']),
 ['id_matricula','fecha_matricula','fecha_inicio','revisar_fecha','dias_anticipacion']])

,id_matricula,fecha_matricula,fecha_inicio,revisar_fecha,dias_anticipacion
0,M0001,2026-01-08,2026-01-25,False,17.0
30,M0031,NaT,2026-01-25,True,NaN
33,M0034,2026-02-05,2026-01-25,True,NaN
36,M0037,2026-01-09,NaT,True,NaN


### Ejercicio 5 · Clasificación por anticipación
Crea `tipo_anticipacion` con tres etiquetas:
- `Anticipada`: 15 días o más.
- `Cercana al inicio`: desde 0 hasta menos de 15 días.
- `No evaluable`: anticipación desconocida.

**Pista conceptual:** comienza con «No evaluable» y asigna las otras etiquetas solo a las filas que cumplen su regla. Una comparación con un nulo no aporta evidencia para clasificar.
<details><summary>Ayuda de sintaxis</summary>

`tabla['categoria'] = pd.Series('No evaluable', index=tabla.index, dtype='string')`  
`tabla.loc[condicion, 'categoria'] = 'Etiqueta'`
</details>

In [22]:
# COMPLETA AQUÍ
UMBRAL_ANTICIPACION = 15
# Crea la columna y asigna etiquetas según las condiciones.
analitica['tipo_anticipacion'] = pd.Series('No evaluable', index=analitica.index, dtype='string')

condicion_anticipada = analitica['dias_anticipacion'].ge(UMBRAL_ANTICIPACION)
condicion_cercana = analitica['dias_anticipacion'].between(0, UMBRAL_ANTICIPACION - 1)

analitica.loc[condicion_anticipada, 'tipo_anticipacion'] = 'Anticipada'
analitica.loc[condicion_cercana, 'tipo_anticipacion'] = 'Cercana al inicio'

In [23]:
assert 'tipo_anticipacion' in analitica
assert analitica.loc[analitica['dias_anticipacion'].isna(),'tipo_anticipacion'].eq('No evaluable').all()
assert analitica.loc[analitica['dias_anticipacion'].ge(15),'tipo_anticipacion'].eq('Anticipada').all()
assert analitica.loc[analitica['dias_anticipacion'].between(0,14),'tipo_anticipacion'].eq('Cercana al inicio').all()
assert analitica['tipo_anticipacion'].eq('No evaluable').sum() == 3
display(analitica['tipo_anticipacion'].value_counts(dropna=False))

,count
tipo_anticipacion,
Cercana al inicio,76
Anticipada,40
No evaluable,3


### Ejercicio 6 · Mes de matrícula
Crea `mes_matricula` como periodo mensual usando `.dt.to_period('M')`. Conserva los faltantes temporales y la columna `periodo`, que describe el semestre académico.

**Pista conceptual:** una fecha identifica un día; un periodo mensual identifica un intervalo. Año y mes juntos distinguen enero de diferentes años.
<details><summary>Ayuda de sintaxis</summary>

`tabla['mes'] = tabla['fecha'].dt.to_period('M')`
</details>

La fecha contradictoria de M0034 permite conocer el mes registrado, aunque su anticipación esté pendiente. Cada análisis requiere sus propios campos válidos.

In [24]:
# COMPLETA AQUÍ
# Crea el periodo mensual sin reemplazar el periodo académico.
analitica['mes_matricula'] = analitica['fecha_matricula'].dt.to_period('M')

In [25]:
assert 'mes_matricula' in analitica
pd.testing.assert_series_equal(analitica['mes_matricula'],base['fecha_matricula'].dt.to_period('M'),check_names=False)
pd.testing.assert_series_equal(analitica['periodo'],base['periodo'])
display(analitica[['id_matricula','fecha_matricula','mes_matricula','periodo']].head())
print('Mes pendiente:',analitica['mes_matricula'].isna().sum())

,id_matricula,fecha_matricula,mes_matricula,periodo
0,M0001,2026-01-08,2026-01,2026-1
1,M0002,2026-01-12,2026-01,2026-1
2,M0003,2026-01-07,2026-01,2026-1
3,M0004,2026-01-07,2026-01,2026-1
4,M0005,2026-01-05,2026-01,2026-1


Mes pendiente: 1


## 5. Una selección con propósito
### Ejercicio 7 · Segmento para comunicación
Selecciona matrículas **activas** y **cercanas al inicio**, mostrando identificador de matrícula, estudiante, programa, sede, rango de edad y días de anticipación. No descartes una matrícula solo porque le falte edad o valor monetario.

**Pista conceptual:** combina condiciones con `&` y usa `.loc`. Cada condición debe tener sus propios paréntesis.
<details><summary>Ayuda de sintaxis</summary>

`segmento = tabla.loc[(tabla['campo_a'].eq('valor')) & (tabla['campo_b'].eq('otro')), columnas].copy()`
</details>

In [26]:
# COMPLETA AQUÍ
# Guarda la selección en segmento y muestra el resultado.
columnas_seleccionadas = ['id_matricula', 'id_estudiante', 'programa', 'sede', 'rango_edad', 'dias_anticipacion']
condicion_filtro = (analitica['estado'].eq('Activa')) & (analitica['dias_anticipacion'].between(0, 14))
segmento = analitica.loc[condicion_filtro, columnas_seleccionadas].copy()
display(segmento)

,id_matricula,id_estudiante,programa,sede,rango_edad,dias_anticipacion
6,M0007,E007,Desarrollo Web,Robledo,25–34,7.0
9,M0010,E010,Marketing,Robledo,35–44,12.0
15,M0016,E016,NaN,Robledo,35–44,9.0
21,M0022,E022,Marketing,Robledo,35–44,6.0
24,M0025,E025,Analítica,Robledo,25–34,5.0
27,M0028,E028,Administración,Robledo,45–59,6.0
39,M0040,E040,Administración,Robledo,35–44,6.0
45,M0046,E046,Marketing,Robledo,45–59,5.0
54,M0055,E055,Desarrollo Web,Robledo,25–34,7.0
57,M0058,E058,Marketing,Robledo,45–59,11.0


In [27]:
esperadas = analitica.loc[analitica['estado'].eq('Activa') & analitica['dias_anticipacion'].between(0,14),'id_matricula']
assert set(segmento['id_matricula']) == set(esperadas), 'Revisa ambas condiciones; no añadas exclusiones por otros campos.'
assert set(['id_matricula','id_estudiante','programa','sede','rango_edad','dias_anticipacion']).issubset(segmento.columns)
print('Selección comprobada.')

Selección comprobada.


### Interpretación
Responde con tus palabras:
1. ¿Qué acción podría apoyar este segmento y qué dato adicional necesitarías para realizarla?
2. ¿Por qué el número de matrículas puede diferir del de estudiantes distintos?
3. ¿Por qué un valor neto calculado no demuestra que la institución recibió ese dinero?
4. ¿Qué cambia si el umbral de anticipación pasa de 15 a 10 días?

**Escribe tus respuestas aquí.**

## 6. Preparación reproducible
### Ejercicio 8 · La misma regla para otra ejecución
Completa la función `enriquecer_matriculas(base_entrada, umbral=15)` con los pasos de los ejercicios 1–6. Trabaja únicamente con el argumento y la copia local `resultado`. No dependas de la variable global `analitica`.

**Reto de autonomía:** reutiliza tus operaciones y cambia el nombre del DataFrame. El parámetro `umbral` debe gobernar ambas condiciones de anticipación. Puedes probar la función con 10 sin alterar las demás variables.

<details><summary>Ayuda para organizar la función</summary>

Dentro de la función: copia la entrada, renombra, calcula, categoriza, trabaja fechas y devuelve el resultado. Mantén todos esos pasos con cuatro espacios de sangría. Usa `resultado` y el parámetro `umbral`.
</details>

In [33]:
def enriquecer_matriculas(base_entrada, umbral=15):
    resultado = base_entrada.copy(deep=True)

    # Ejercicio 1: Renombrar columna
    resultado = resultado.rename(columns={'valor_matricula': 'valor_bruto_matricula'})

    # Ejercicio 2: Cálculos vectorizados de descuento y neto
    resultado['valor_descuento'] = (resultado['valor_bruto_matricula'] * resultado['descuento_pct'] / 100).round(2)
    resultado['valor_neto_matricula'] = (resultado['valor_bruto_matricula'] - resultado['valor_descuento']).round(2)

    # Ejercicio 3: Rangos de edad
    cortes_edad = [16, 25, 35, 45, 60, 81]
    etiquetas_edad = ['16–24', '25–34', '35–44', '45–59', '60–80']
    resultado['rango_edad'] = pd.cut(resultado['edad'], bins=cortes_edad, labels=etiquetas_edad, right=False).astype('string').fillna('Sin información')

    # Ejercicio 4: Días de anticipación y validez
    duracion_local = (resultado['fecha_inicio'] - resultado['fecha_matricula']).dt.days
    resultado['dias_anticipacion'] = duracion_local.mask(resultado['revisar_fecha'])

    # Ejercicio 5: Clasificación por anticipación usando el parámetro 'umbral'
    resultado['tipo_anticipacion'] = pd.Series('No evaluable', index=resultado.index, dtype='string')
    condicion_anticipada = resultado['dias_anticipacion'].ge(umbral)
    condicion_cercana = resultado['dias_anticipacion'].between(0, umbral - 1)
    resultado.loc[condicion_anticipada, 'tipo_anticipacion'] = 'Anticipada'
    resultado.loc[condicion_cercana, 'tipo_anticipacion'] = 'Cercana al inicio'

    # Ejercicio 6: Mes de matrícula
    resultado['mes_matricula'] = resultado['fecha_matricula'].dt.to_period('M')

    return resultado

In [34]:
salida_1 = enriquecer_matriculas(base, umbral=15)
salida_2 = enriquecer_matriculas(base, umbral=15)
pd.testing.assert_frame_equal(salida_1, salida_2)
pd.testing.assert_frame_equal(salida_1, analitica)
pd.testing.assert_frame_equal(base, base_control)
assert len(salida_1) == len(base) and salida_1['id_matricula'].is_unique
assert salida_1['dias_anticipacion'].dropna().ge(0).all()
prueba_10 = enriquecer_matriculas(base, umbral=10)
assert prueba_10.loc[prueba_10['dias_anticipacion'].ge(10),'tipo_anticipacion'].eq('Anticipada').all()
assert prueba_10.loc[prueba_10['dias_anticipacion'].between(0,9),'tipo_anticipacion'].eq('Cercana al inicio').all()
print('Proceso reproducible, entrada conservada y parámetro funcional.')

Proceso reproducible, entrada conservada y parámetro funcional.


## 7. Exportación y registro de reglas
Exporta la salida, el segmento y los registros separados. En Colab los archivos quedan en tu carpeta de Drive. Los CSV no conservan los tipos de Pandas: al leer nuevamente, debes convertir fechas y periodos.

| Variable | Significado / unidad | Faltantes y criterio |
|---|---|---|
| valor_bruto_matricula | Valor original válido, COP | Desconocido si inválido |
| valor_descuento | Monto del descuento, COP | Requiere bruto y porcentaje válidos |
| valor_neto_matricula | Bruto menos descuento, COP | No representa recaudo |
| rango_edad | Edad segmentada | Sin información si edad desconocida |
| dias_anticipacion | Inicio menos matrícula, días | Desconocido si revisar_fecha |
| tipo_anticipacion | Segmentación temporal | Anticipada ≥15; cercana 0–14; no evaluable |
| mes_matricula | Intervalo año-mes | NaT si fecha no interpretable |

In [36]:
SALIDAS = Path('salidas_clase_8')
SALIDAS.mkdir(parents=True,exist_ok=True)
salida_1.to_csv(SALIDAS/'Matriculas_enriquecidas.csv',index=False,encoding='utf-8-sig')
segmento.to_csv(SALIDAS/'Segmento_matriculas_cercanas.csv',index=False,encoding='utf-8-sig')
copias.to_csv(SALIDAS/'Matriculas_copias_exactas.csv',index=False,encoding='utf-8-sig')
pendientes.to_csv(SALIDAS/'Matriculas_pendientes_llave.csv',index=False,encoding='utf-8-sig')
print('Archivos guardados en:',SALIDAS)

Archivos guardados en: salidas_clase_8


## Si algo falla
| Mensaje o síntoma | Revisión |
|---|---|
| FileNotFoundError | Monta Drive. Revisa la ruta CARPETA y el nombre exacto. |
| KeyError | Comprueba `analitica.columns` y si ya ejecutaste el renombrado. |
| Error con `.dt` | Revisa que la preparación convirtió fechas y que no reasignaste texto. |
| TypeError al rellenar categorías | Convierte la salida de cut a `string` antes de fillna. |
| AssertionError | Lee el mensaje, inspecciona la fila o variable y corrige la regla. |
| NameError | Ejecuta las celdas previas y verifica la escritura del nombre. |
| Resultado cambia entre ejecuciones | Reconstruye desde df_original. Evita depender de celdas fuera de orden. |

## Cierre
Revisa tus respuestas y los controles. Conserva el notebook y la base enriquecida: en la próxima clase podrán servir para construir indicadores con `groupby`.
La encuesta de avance del proyecto se responde después de la sesión. Proyecto individual o equipos de máximo cuatro, presentación de máximo diez minutos el 27 de noviembre.

## Referencias oficiales
- [Pandas: rename](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html)
- [Pandas: map](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html)
- [Pandas: cut](https://pandas.pydata.org/docs/reference/api/pandas.cut.html)
- [Pandas: fechas y periodos](https://pandas.pydata.org/docs/user_guide/timeseries.html)
- [Colab: archivos y Google Drive](https://colab.research.google.com/notebooks/io.ipynb)